# 16 · Matrix Calculus and Automatic Differentiation

- **Level:** Advanced (bridge from engineering maths to deep learning)
- **Typical semester:** not usually taught as a course; learned alongside ML in Semester 5–8
- **Prerequisites:** 02, 06
- **Used later in:** deriving every gradient in 17–24, and building an autograd engine (`lipipy.dl`).


## 1. Layout conventions

Using the denominator (gradient) layout: for scalar $f$ and vector $x\in\mathbb R^n$, $\nabla_xf\in\mathbb R^n$; for scalar $f$ and matrix $X$, $\frac{\partial f}{\partial X}$ has the same shape as $X$.

Jacobian of $y = f(x)$, $y\in\mathbb R^m$: $J_{ij} = \frac{\partial y_i}{\partial x_j}\in\mathbb R^{m\times n}$. Hessian $H_{ij} = \frac{\partial^2f}{\partial x_i\partial x_j}$.


## 2. Vector derivatives

$$\nabla_x(a^\top x) = a, \qquad \nabla_x(x^\top Ax) = (A + A^\top)x \;(= 2Ax \text{ if } A \text{ symmetric})$$
$$\nabla_x\|x\|_2^2 = 2x, \qquad \nabla_x\|x\|_2 = \frac{x}{\|x\|_2}, \qquad \nabla_x\|Ax - b\|^2 = 2A^\top(Ax - b)$$
$$\frac{\partial(Ax)}{\partial x} = A, \qquad \nabla_x^2(x^\top Ax) = A + A^\top$$


## 3. Matrix derivatives

$$\frac{\partial\,\text{tr}(AX)}{\partial X} = A^\top, \qquad \frac{\partial\,\text{tr}(X^\top A)}{\partial X} = A, \qquad \frac{\partial\,\text{tr}(X^\top AX)}{\partial X} = (A + A^\top)X$$
$$\frac{\partial\,a^\top Xb}{\partial X} = ab^\top, \qquad \frac{\partial\|X\|_F^2}{\partial X} = 2X, \qquad \frac{\partial\|AXB - C\|_F^2}{\partial X} = 2A^\top(AXB - C)B^\top$$
$$\frac{\partial\log\det X}{\partial X} = X^{-\top}, \qquad \frac{\partial\det X}{\partial X} = \det(X)X^{-\top}, \qquad \frac{\partial X^{-1}}{\partial t} = -X^{-1}\frac{\partial X}{\partial t}X^{-1}$$


## 4. Differentials (the easiest way to derive gradients)

Write $df$, use $d(XY) = dX\,Y + X\,dY$, $d(X^{-1}) = -X^{-1}dX\,X^{-1}$, $d\log\det X = \text{tr}(X^{-1}dX)$, then arrange $df = \text{tr}(G^\top dX)$; the gradient is $G$.

Example: $f = \|Xw - y\|^2$, $df = 2(Xw - y)^\top X\,dw$, so $\nabla_wf = 2X^\top(Xw - y)$.


## 5. Gradients of common ML building blocks

**Linear layer** $Y = XW + b$ ($X\in\mathbb R^{B\times d_{in}}$), upstream gradient $G = \frac{\partial L}{\partial Y}$:
$$\frac{\partial L}{\partial W} = X^\top G, \qquad \frac{\partial L}{\partial b} = \mathbf 1^\top G, \qquad \frac{\partial L}{\partial X} = GW^\top$$

**Elementwise activation** $y = \sigma(x)$: $\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y}\odot\sigma'(x)$

**Softmax + cross-entropy** with one-hot $y$: $L = -\sum y_k\log p_k$, $p = \text{softmax}(z)$ ⇒ $\frac{\partial L}{\partial z} = p - y$

**Sigmoid + binary cross-entropy:** $\frac{\partial L}{\partial z} = \sigma(z) - y$

**MSE** $L = \frac1n\|\hat y - y\|^2$: $\frac{\partial L}{\partial\hat y} = \frac2n(\hat y - y)$

**Layer normalization** $\hat x = \frac{x - \mu}{\sigma}$, $y = \gamma\odot\hat x + \beta$ over $d$ features:
$$\frac{\partial L}{\partial x} = \frac{1}{\sigma}\left(g - \frac1d\sum_jg_j - \hat x\frac1d\sum_jg_j\hat x_j\right), \quad g = \gamma\odot\frac{\partial L}{\partial y}$$
(batch norm is the same formula taken over the batch dimension)

**Scaled dot-product attention** $S = \frac{QK^\top}{\sqrt d}$, $P = \text{softmax}(S)$ row-wise, $O = PV$:
$$dV = P^\top dO, \qquad dP = dO\,V^\top, \qquad dS_{ij} = P_{ij}\left(dP_{ij} - \sum_kP_{ik}dP_{ik}\right), \qquad dQ = \frac{dS\,K}{\sqrt d}, \qquad dK = \frac{dS^\top Q}{\sqrt d}$$

**Convolution:** the gradient w.r.t. the input is a "full" convolution of the upstream gradient with the flipped kernel; the gradient w.r.t. the kernel is a cross-correlation of the input with the upstream gradient.

**RNN** $h_t = \tanh(W_hh_{t-1} + W_xx_t)$: backpropagation through time multiplies Jacobians $\prod_t\text{diag}(1 - h_t^2)W_h$, which explodes or vanishes depending on the spectral radius of $W_h$.


## 6. Automatic differentiation

**Forward mode:** propagate tangents $\dot v = \sum_u\frac{\partial v}{\partial u}\dot u$ (dual numbers $a + b\varepsilon$, $\varepsilon^2 = 0$). Cost: one pass per input direction. Best when inputs are few.

**Reverse mode (backpropagation):** propagate adjoints $\bar u = \sum_{v:\,u\to v}\bar v\frac{\partial v}{\partial u}$ from the output back. Cost: one pass per output. Best for scalar losses with millions of parameters.

Vector-Jacobian product (VJP) $v^\top J$ is reverse mode; Jacobian-vector product (JVP) $Jv$ is forward mode. Hessian-vector product $Hv = \nabla_x(\nabla_xf\cdot v)$ costs about two gradients.

**Gradient checking:** compare with $\frac{f(\theta + \epsilon e_i) - f(\theta - \epsilon e_i)}{2\epsilon}$; relative error $\frac{|g_a - g_n|}{|g_a| + |g_n|}$ should be about $10^{-7}$ in float64.

**Memory:** reverse mode stores activations. Gradient checkpointing trades compute for memory (store every $\sqrt n$ layers).


## 7. Gradient estimators for stochastic objectives

**Score-function / REINFORCE estimator:** $\nabla_\theta\mathbb E_{p_\theta}[f(x)] = \mathbb E_{p_\theta}[f(x)\nabla_\theta\log p_\theta(x)]$ (high variance; use baselines)

**Reparameterization (pathwise) estimator:** $x = g_\theta(\epsilon)$, $\epsilon\sim p(\epsilon)$ ⇒ $\nabla_\theta\mathbb E[f(x)] = \mathbb E_\epsilon[\nabla_\theta f(g_\theta(\epsilon))]$ (low variance; needs differentiable $f$)

**Straight-through estimator:** use the identity gradient through a non-differentiable op (quantization, VQ-VAE).

**Implicit function theorem:** if $F(x, y^*(x)) = 0$, then $\frac{dy^*}{dx} = -\left[\frac{\partial F}{\partial y}\right]^{-1}\frac{\partial F}{\partial x}$ (deep equilibrium models, differentiable optimization layers).

**Adjoint method (neural ODEs):** $\frac{da}{dt} = -a^\top\frac{\partial f}{\partial z}$, $\frac{dL}{d\theta} = -\int_{t_1}^{t_0}a^\top\frac{\partial f}{\partial\theta}dt$, which gives memory cost constant in the number of solver steps.
